# 04. Alignments (вся история, без окна)

Отдельный ноутбук, чтобы самый долгий расчёт шёл независимо от остальных. Заменяет `run_split.py`, вторую половину `petri_states_eval_*.ipynb`, `retry_timeouts.py`, `fix_timeouts_051126.py` и `im_timeouts.py`.

**Сначала выполнить `03_tbr_precision.ipynb`.** Этот ноутбук в обеих частях можно запускать одновременно: процессы лёгкие, журнал они не строят, папки результатов разные.

Как идёт расчёт (`common/align_parallel.py`): свободный процесс берёт следующий вариант трассы любой модели; каждый результат сразу записан. *Kernel → Interrupt* останавливает всё, запуск ячейки заново продолжает с места.

Лимит на вариант — **20 с** (как в прежнем расчёте этой части). Лимит считается по настенным часам: если процессор перегружен, тайм-аутов будет больше, поэтому `WORKERS` не стоит ставить выше числа свободных ядер.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "common"))
import settings as cfg

PART = "all"                         # "2013" — с окном, "all" — без окна

import pandas as pd
import align_parallel as ap

WORKERS = cfg.N_WORKERS                  # процессов в этом ноутбуке; см. комментарий в settings.py
CONFIGS = cfg.EVAL_CONFIGS               # если их несколько, считаются вперемешку одним набором процессов
SCRIPT = cfg.COMMON / "align_parallel.py"
FIX = cfg.COMMON / "fix_timeouts.py"

## Шаг 1. Alignments на всём test, лимит 20 с

In [2]:
ap.launch([SCRIPT, PART, "--configs", *CONFIGS, "--workers", WORKERS])

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\align_parallel.py all --configs long180_v2 long180_v1 long365_v1 --workers 3


01:45:54 часть all: заданий 9, процессов 3


01:45:54 проверка easy soundness: 9 сетей (лимит 600 с на сеть)


01:45:57   long180_v1/DFG_p0.05: easy_sound = True


01:45:57   long180_v1/HM_dep0.95: easy_sound = False


01:45:57   long180_v1/SplitMiner: easy_sound = False


... (116 outputs left out)

02:26:18 long365_v1/DFG_p0.05 готово: fitness 0.919 | тайм-аут трасс 0 из 2295


02:26:18 long365_v1/SplitMiner готово: fitness 0.114 | тайм-аут трасс 0 из 2295


02:26:18 long365_v1/IM_noise0.2 готово: fitness 0.9375 | тайм-аут трасс 9 из 2295


02:26:18 ALIGN_DONE, 40.4 мин


## Шаг 2. Досчёт тайм-аутов, LONG = 180

DFG p0.05 — точный расчёт (минуты); IM 0.2 — варианты, идеальные по TBR, и лимит 600 с на остальные (самый долгий шаг).

Итог: `out/long180_v2/split_eval_timeouts_fixed.csv`.

In [3]:
for config in CONFIGS:
    if config.startswith("long180"):
        ap.launch([FIX, PART, f"--workers={WORKERS}", f"--config={config}"])

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\fix_timeouts.py all --workers=3 --config=long180_v2


02:26:21 all: вариантов в test 1,630


02:26:21 all DFG_p0.05: тайм-аутов 0 вариантов (0 трасс)


02:26:21 шаг dfg: 0.0 мин


experiment       model  timeout_before  timeout_after  fitness_align_before  fitness_align_after  f1_align_before  f1_align_after


       all   DFG_p0.05               0              0                0.9211               0.9211           0.7409          0.7409


... (37 outputs left out)

       all    DFG_p0.0               0              0                1.0000               1.0000           0.4163          0.4163


       all IM_noise0.2               8              0                0.9392               0.9387           0.6117          0.6115


02:27:58 итог -> split_eval_timeouts_fixed.csv


FIX_DONE


## Шаг 3. Досчёт тайм-аутов варианта 1, `long365_v1`, лимит 120 с

Выполняется только при `RUN_V1 = True`.

Варианты, не уложившиеся в лимит шага 1, для моделей из `cfg.PARTS[PART]["retry_models"]`. В прежнем расчёте этот шаг был остановлен на середине и итога не дал. Можно прервать в любой момент: `07_results_long365.ipynb` покажет, сколько досчитано. Итог: `out/long365_v1/split_eval_retry*s.jsonl`.

In [4]:
if cfg.RUN_V1:
    ap.launch([SCRIPT, PART, "--configs", "long365_v1", "--retry", "--workers", WORKERS])
else:
    print("шаг выключен: RUN_V1 = False в common/settings.py")

D:\Git\diploma\.venv\Scripts\python.exe -u D:\diploma\jupyter lab\петри_пересчет\common\align_parallel.py all --configs long365_v1 --retry --workers 3


02:27:59 часть all: заданий 1, процессов 3


02:27:59   long365_v1/IM_noise0.2: вариантов 1,626, уже готово 0, осталось 9, лимит 120 с


02:29:34 выровнено 9 / 9, 1.6 мин


02:29:34 long365_v1/IM_noise0.2 готово: fitness 0.937 | тайм-аут трасс 0 из 2295


02:29:34 ALIGN_DONE, 1.6 мин


## Шаг 4. Сколько посчитано

Можно запускать в любой момент из отдельной консоли ноутбука: читает только контрольные точки.

In [5]:
rows = []
for config in CONFIGS:
    keys, counts = ap.load_variants(PART, config)
    d = cfg.out_dir(PART, config)
    finished = {r["model"]: r for r in ap.jl_read(d / "split_eval.jsonl")}
    for name in cfg.EVAL_MODELS[config]:
        ck = ap.jl_read(d / "align_ckpt" / f"{name}_align.jsonl")
        rows.append({"конфигурация": config, "модель": name, "вариантов": len(keys), "готово": len(ck),
                     "тайм-аутов (варианты)": sum(r["timeout"] for r in ck), "модель закончена": name in finished,
                     "fitness": finished.get(name, {}).get("fitness_align_test"), "часов процессора": round(sum(r["sec"] for r in ck) / 3600, 2)})
pd.DataFrame(rows).round(4)

,конфигурация,модель,вариантов,готово,тайм-аутов (варианты),модель закончена,fitness,часов процессора
0,long180_v2,DFG_p0.05,1630,1630,0,True,0.9211,0.09
1,long180_v2,DFG_p0.0,1630,1630,0,True,1.0000,0.02
2,long180_v2,HM_dep0.95,1630,0,0,True,NaN,0.00
3,long180_v2,SplitMiner,1630,1630,0,True,0.2167,0.04
4,long180_v2,IM_noise0.2,1630,1630,9,True,0.9391,0.90
5,long180_v1,DFG_p0.05,1630,1630,0,True,0.9211,0.09
6,long180_v1,DFG_p0.0,1630,1630,0,True,1.0000,0.02
7,long180_v1,HM_dep0.95,1630,0,0,True,NaN,0.00
8,long180_v1,SplitMiner,1630,0,0,True,NaN,0.00
9,long180_v1,IM_noise0.2,1630,1630,8,True,0.9392,0.87
